# LLM A pilot — bake-off

LLM A: `query + qrels gold doc + corpus stats hints -> predicted route rank order, or abstain`
(decision 2/3/12/A3). It **predicts**, never observes retrieval (M1) — its disagreement with the
spine means mechanism and outcome diverge (distractors, or odd qrels), never "the label is wrong."

**Three candidates, same draw, real calls:**

| role | model |
|---|---|
| primary | `openai/gpt-5.6-luna` |
| bake-off alt | `deepseek/deepseek-v4-flash` |
| bake-off alt | `google/gemini-3.1-flash-lite` |

All via OpenRouter's REST API directly (not litellm — its cost calculator doesn't know these models
yet; OpenRouter's own `usage.cost` is authoritative and doesn't need it to). Real dollars, expect well
under a cent per model. This is **not** the pre-registered 500-row calibration spike
(route-label-sourcing.md) — n here is far too small for that gate; it proves the pipeline, the price,
and which candidate is worth spending the real spike's budget on.

In [1]:
import os
import re

import pandas as pd
import requests
from dotenv import load_dotenv

from composition.pool_v3 import LabelledPool
from hybrid_search_rrf_dataset.router import DATA_DIR

load_dotenv()
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPEN_ROUTER_API_KEY")
assert OPENROUTER_KEY, "no OpenRouter key in .env"
COMPLETIONS_URL = "https://openrouter.ai/api/v1/chat/completions"

MODELS = {
    "primary": "openai/gpt-5.6-luna",
    "deepseek": "deepseek/deepseek-v4-flash",
    "gemini-flash-lite": "google/gemini-3.1-flash-lite",
}
pd.set_option("display.width", 170)

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
def complete(model: str, system: str, user: str, *, max_tokens: int = 120) -> tuple[str, dict]:
    """Real call, real cost read from OpenRouter's own `usage.cost` — not litellm's
    static price list, which does not know brand-new model ids yet (verified: it
    raises 'This model isn't mapped yet' on `openai/gpt-5.6-luna`).

    `reasoning: {effort: none}` is required, not optional: `gpt-5.6-luna` and
    `deepseek-v4-flash` reason by default and, on the real (long) prompt below,
    spent the ENTIRE `max_tokens` budget on hidden reasoning tokens — verified:
    `finish_reason='length'`, `content=None`, `reasoning_tokens=64`, still
    CHARGED ($0.0003454 for zero visible output). Without this, a full-pool run
    on the primary model would silently return empty answers while billing for
    them. `gemini-3.1-flash-lite` does not reason by default here, but the flag
    is harmless to send regardless. `max_tokens` sized for ORDER + WHY (two short
    lines); with effort:none there is no hidden reasoning to eat it."""
    r = requests.post(
        COMPLETIONS_URL,
        headers={"Authorization": f"Bearer {OPENROUTER_KEY}", "Content-Type": "application/json"},
        json={"model": model, "max_tokens": max_tokens, "usage": {"include": True},
              "reasoning": {"effort": "none"},
              "messages": [{"role": "system", "content": system},
                           {"role": "user", "content": user}]},
        timeout=30,
    )
    r.raise_for_status()
    body = r.json()
    content = body["choices"][0]["message"]["content"]
    return (content or "").strip(), body.get("usage", {})

## 1. Draw — tiny, stratified, real data

In [3]:
SEED = 0
N_PER_KIND = 4
LANE = "crumb-legal-qa"

pool = LabelledPool()
classified = pool.classify(pool.labels())
lane_rows = classified[classified["dataset"] == LANE]
draws = [lane_rows[lane_rows["kind"] == k].sample(min(N_PER_KIND, len(lane_rows[lane_rows["kind"] == k])), random_state=SEED)
         for k in ("decisive", "undecisive", "genuine_tie", "all_zero")]
drawn = pd.concat(draws, ignore_index=True)
print(f"drew {len(drawn)} rows: " + drawn["kind"].value_counts().to_dict().__repr__())

drew 16 rows: {'decisive': 4, 'undecisive': 4, 'genuine_tie': 4, 'all_zero': 4}


## 2. Build the real prompt — gold doc + corpus stats, decision 12's abstain token

In [10]:
qrels = pd.read_parquet(DATA_DIR / LANE / "qrels.parquet").astype({"query_id": str, "doc_id": str})
corpus = pd.read_parquet(DATA_DIR / LANE / "corpus.parquet")

GOLD_CHARS = 1200
"""The repo's own convention (`ParentPool.gold_text(chars=1200)`,
`augmentation/parents.py:105`) and the basis of the measured per-row cost below.
Stated rather than silent, because it is a real limitation: 85% of
crumb-legal-qa docs exceed 1200 chars (median 1940), so LLM A judges most rows
from a truncated gold doc. Defensible here — LLM A's question (does this
query<->answer pair need lexical or semantic matching?) is usually decidable
from a document's opening — but a gold doc whose distinguishing content sits
past the cutoff is a source of error this pilot cannot see. Raising it raises
the measured cost roughly proportionally on the input side."""

corpus_text = {
    str(r.doc_id): (f"{r.title}\n{r.text}" if "title" in corpus.columns else str(r.text)).strip()[:GOLD_CHARS]
    for r in corpus.itertuples()
}
stats = pd.read_parquet(DATA_DIR / "route_labels" / "query_corpus_stats.parquet").astype({"query_id": str})

INSTRUCTION = (
    "You predict which retrieval strategy will best answer a search query against a specific "
    "document collection: dense (semantic embedding search), sparse (BM25 lexical search), or "
    "hybrid (fused, reciprocal rank fusion of both). You do not see any retrieval results — "
    "reason from the query, the one document known to answer it, and statistics about the target "
    "collection's vocabulary.\n\n"
    "Reply with EXACTLY these two lines, or the ABSTAIN line:\n"
    "ORDER: <best> > <middle> > <worst>   (using dense, sparse, hybrid)\n"
    "WHY: <one short clause naming what in the query, answer doc, or collection stats drove the order>\n"
    "ABSTAIN: <one short reason>\n"
    "Use ABSTAIN when the query gives you no real basis to prefer one strategy over another."
)


def build_prompt(row) -> tuple[str, str] | tuple[None, None]:
    """-> (prompt, stats_line); (None, None) when the query has no judged gold doc.
    `stats_line` is returned so the caller can store the numbers the model saw
    and later check its WHY against them."""
    judged = qrels.loc[
        (qrels["query_id"] == str(row.query_id)) & (qrels["relevance"] >= row.min_relevance),
        "doc_id",
    ]
    if judged.empty:
        return None, None
    gold_text = corpus_text.get(str(judged.iloc[0]), "")
    s = stats.loc[stats["query_id"] == str(row.query_id)]
    stats_line = (
        "unavailable" if s.empty else
        ", ".join(f"{c}={s.iloc[0][c]:.3g}" for c in
                  ("avg_idf", "max_idf", "oov_share", "vocab_overlap", "mean_pmi", "collection_size"))
    )
    prompt = (
        f"Query: {row.query}\n\n"
        f"Known answer document:\n{gold_text}\n\n"
        f"Target collection statistics for this query: {stats_line}"
    )
    return prompt, stats_line

## 3. LIVE — spends real money, once per candidate model

Expected: a few hundred input tokens + ~20-60 output tokens per row per model — well under $0.001/row
on any of the three. Run this cell explicitly; nothing above it touches the network.

In [11]:
RUN_LIVE = True  # flip to True to actually spend

results = []
if RUN_LIVE:
    for role, model in MODELS.items():
        for row in drawn.itertuples():
            prompt, stats_line = build_prompt(row)
            if prompt is None:
                continue
            text, usage = complete(model, INSTRUCTION, prompt)
            results.append({
                "role": role, "model": model, "dataset": row.dataset, "query_id": row.query_id,
                "kind": row.kind, "llm_a_raw": text, "stats_line": stats_line, "cost_usd": usage.get("cost", 0),
                "prompt_tokens": usage.get("prompt_tokens"), "completion_tokens": usage.get("completion_tokens"),
            })
    results = pd.DataFrame(results)
    print(results.groupby("role")["cost_usd"].agg(["count", "sum", "mean"]))
else:
    print("RUN_LIVE is False — flip it above to actually call the API")

                   count       sum      mean
role                                        
deepseek              16  0.001089  0.000068
gemini-flash-lite     16  0.003265  0.000204
primary               16  0.002394  0.000150


## 4. Parse + compare against the spine (informal — n far too small for the real gate)

In [12]:
ROUTE_MAP = {"dense": "dense_only", "sparse": "sparse_only", "hybrid": "pure_rrf"}
ROUTE_LABEL = {v: k for k, v in ROUTE_MAP.items()}  # winner id -> friendly name


def parse(raw: str) -> tuple[list[str] | None, str | None, str | None]:
    """-> (rank_order, why, abstain_reason). A well-formed answer is either
    (order, why, None) or (None, None, abstain_reason)."""
    order = why = abstain = None
    for line in raw.splitlines():
        line = line.strip()
        if m := re.match(r"ORDER:\s*(.+)", line, re.IGNORECASE):
            order = [t.strip().lower() for t in re.split(r">", m.group(1))]
        elif m := re.match(r"WHY:?\s*(.*)", line, re.IGNORECASE):
            why = m.group(1).strip() or None
        elif m := re.match(r"ABSTAIN:?\s*(.*)", line, re.IGNORECASE):
            abstain = m.group(1).strip() or "(no reason given)"
    if order is None and abstain is None:
        abstain = f"UNPARSEABLE: {raw!r}"
    return order, why, abstain


if RUN_LIVE and len(results):
    parsed = results["llm_a_raw"].map(parse)
    results["llm_a_top"] = parsed.map(lambda p: p[0][0] if p[0] else None)
    results["llm_a_why"] = parsed.map(lambda p: p[1])
    results["llm_a_abstain"] = parsed.map(lambda p: p[2])
    results["llm_a_top_route"] = results["llm_a_top"].map(ROUTE_MAP)

    decisive = results[results["kind"] == "decisive"].merge(
        classified[["dataset", "query_id", "winner"]], on=["dataset", "query_id"], how="left",
    )
    for role, group in decisive.groupby("role"):
        abstain_rate = group["llm_a_top_route"].isna().mean()
        answered = group[group["llm_a_top_route"].notna()]
        # abstain is zero weight (decision 12), never a vote against — excluded
        # from the denominator, not counted as disagreement
        line = f"{role:20s} decisive rows: {len(group)}  abstained: {abstain_rate:.0%}"
        if len(answered):
            agree = (answered["llm_a_top_route"] == answered["winner"]).mean()
            line += f"  agreement on answered ({len(answered)}): {agree:.0%}"
        else:
            line += "  (none answered)"
        print(line)

deepseek             decisive rows: 4  abstained: 0%  agreement on answered (4): 25%
gemini-flash-lite    decisive rows: 4  abstained: 0%  agreement on answered (4): 25%
primary              decisive rows: 4  abstained: 0%  agreement on answered (4): 25%


## 4b. Decisive rows — does LLM A hold a different opinion, and why?

These are rows retrieval already settled (`kind == decisive`, a clear spine winner). A disagreement here is the case worth reading: LLM A *predicts* a different best route than the one measurement observed. Per M1 that never means the label is wrong — it means mechanism and outcome diverge (a distractor in the gold doc, odd qrels). `verdict` splits agree / differ / abstain (abstain is zero-weight, not a vote against). For every `differ`, the block below prints the model's own `why` next to the `saw` stats line it was given — so you can judge whether the stated basis is real or a confabulation.

In [13]:
if RUN_LIVE and len(results):
    dec = results[results["kind"] == "decisive"].merge(
        classified[["dataset", "query_id", "winner", "query"]],
        on=["dataset", "query_id"], how="left")
    dec["spine"] = dec["winner"].map(ROUTE_LABEL)
    dec["verdict"] = [
        "abstain" if pd.isna(rt) else ("agree" if rt == w else "differ")
        for rt, w in zip(dec["llm_a_top_route"], dec["winner"])
    ]
    dec = dec.sort_values(["verdict", "role"]).reset_index(drop=True)
    print(dec["verdict"].value_counts().to_dict(), "\n")

    scan = dec[["role", "spine", "llm_a_top", "verdict", "query"]].copy()
    scan["query"] = scan["query"].astype(str).str.slice(0, 48)
    print(scan.rename(columns={"llm_a_top": "llm_a"}).to_string(index=False))

    print("\n--- opinions that DIFFER from the spine, in full (read the WHY, check it against `saw`) ---")
    diffs = dec[dec["verdict"] == "differ"]
    if diffs.empty:
        print("  (none — no model disagreed on a decisive row in this draw)")
    for _, r in diffs.iterrows():
        print(f"\n[{r.role}]  spine={r.spine}  ->  llm_a={r.llm_a_top}")
        print(f"  query: {r.query}")
        print(f"  why  : {r.llm_a_why}")
        print(f"  saw  : {r.stats_line}")

{'differ': 9, 'agree': 3} 

             role  spine  llm_a verdict                                            query
         deepseek sparse sparse   agree Is the term 'Execution' used to refer to the ord
gemini-flash-lite sparse sparse   agree Is the term 'Execution' used to refer to the ord
          primary sparse sparse   agree Is the term 'Execution' used to refer to the ord
         deepseek  dense sparse  differ Primary methods of service are defined as those 
         deepseek  dense sparse  differ Primary methods of service are defined as those 
         deepseek  dense sparse  differ Is the term 'Writ for removal' used to refer to 
gemini-flash-lite  dense hybrid  differ Primary methods of service are defined as those 
gemini-flash-lite  dense hybrid  differ Primary methods of service are defined as those 
gemini-flash-lite  dense hybrid  differ Is the term 'Writ for removal' used to refer to 
          primary  dense hybrid  differ Primary methods of service are defined as 

## 4c. Undecided branches — what does LLM A add where measurement is weak or silent?

`undecisive` (narrow spine margin), `genuine_tie`, and `all_zero` (no route retrieved a judged doc) are the rows a label change can actually move — here LLM A is the tie-breaker, not a second-guesser of a clear outcome. Three questions: does it commit to a route or abstain, which way does it break the near-tie versus the spine's weak winner, and do the three models agree *with each other* (a signal that the pick is stable rather than noise)?

In [ ]:
if RUN_LIVE and len(results):
    UND = ["undecisive", "genuine_tie", "all_zero"]
    und = results[results["kind"].isin(UND)].merge(
        classified[["dataset", "query_id", "winner", "query"]],
        on=["dataset", "query_id"], how="left")
    und["pick"] = und["llm_a_top"].fillna("ABSTAIN")

    print("pick distribution per kind per model:\n")
    for kind in UND:
        sub = und[und["kind"] == kind]
        if sub.empty:
            continue
        print(f"  {kind} ({sub['query_id'].nunique()} rows):")
        for role, g in sub.groupby("role"):
            print(f"    {role:20s} {g['pick'].value_counts().to_dict()}")
        print()

    # do the three models land on the same pick for a given row, or scatter?
    pivot = und.pivot_table(index=["kind", "query_id"], columns="role",
                            values="pick", aggfunc="first")
    consensus = (pivot.nunique(axis=1) == 1).sum()
    print(f"cross-model consensus: {consensus}/{len(pivot)} undecided rows got the "
          f"identical pick from all three models\n")

    # undecisive rows carry a real (narrow) spine winner — show confirm vs flip
    uc = und[und["kind"] == "undecisive"].copy()
    if not uc.empty:
        uc["spine"] = uc["winner"].map(ROUTE_LABEL)
        print("--- undecisive: does LLM A confirm or flip the spine's narrow winner? ---")
        for _, r in uc.sort_values(["query_id", "role"]).iterrows():
            move = "confirm" if r["llm_a_top_route"] == r["winner"] else "flip / abstain"
            print(f"\n[{r.role}]  spine≈{r.spine}  ->  llm_a={r.pick}  ({move})")
            print(f"  query: {str(r.query)[:60]}")
            print(f"  why  : {r.llm_a_why or r.llm_a_abstain}")

## 5. Real cost per candidate, extrapolated

In [7]:
if RUN_LIVE and len(results):
    summary = results.groupby("role").agg(
        n=("cost_usd", "count"), per_row=("cost_usd", "mean"),
        mean_prompt_tok=("prompt_tokens", "mean"), mean_completion_tok=("completion_tokens", "mean"),
    )
    summary["extrapolated_224910"] = summary["per_row"] * 224_910
    print(summary.to_string())
    print("\nplan's hand estimate was: ~$127 (haiku)")

                    n   per_row  mean_prompt_tok  mean_completion_tok  extrapolated_224910
role                                                                                      
deepseek           16  0.000050         501.7500              10.6875            11.146436
gemini-flash-lite  16  0.000149         529.9375              11.1250            33.550246
primary            16  0.000114         501.8750              11.0000            25.544153

plan's hand estimate was: ~$127 (haiku)
